# Handling Large Multi-Modal Payloads in AgentCore Runtime

## Overview

This tutorial demonstrates how Amazon Bedrock AgentCore Runtime handles large payloads up to 100MB, including multi-modal content such as Excel files and images. AgentCore Runtime is designed to process rich media content and large datasets seamlessly.

### Tutorial Details

|Information| Details|
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Large Payload & Multi-Modal Processing|
| Agent type          | Single         |
| Agentic Framework   | Strands Agents |
| LLM model           | Anthropic Claude Haiku 4.5 |
| Tutorial components | Large File Processing, Image Analysis, Excel Data Processing |
| Tutorial vertical   | Data Analysis & Multi-Modal AI                                                   |
| Example complexity  | Intermediate                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK|

### Key Features

* **Large Payload Support**: Process files up to 100MB in size
* **Multi-Modal Processing**: Handle Excel files, images, and text simultaneously
* **Data Analysis**: Extract insights from structured data and visual content
* **Base64 Encoding**: Secure transmission of binary data through JSON payloads

## Prerequisites

* Python 3.10+
* AWS credentials configured
* Docker running
* Sample Excel file and image for testing

In [ ]:
!uv pip install --force-reinstall -U -r requirements.txt --quiet

## Create Sample Data Files

Let's create sample Excel and image files to demonstrate large payload handling:

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
user_name = os.getenv("USER_NAME")
if not user_name:
    raise ValueError("USER_NAME environment variable is not set. Please set it in the .env file.")


In [2]:
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os

# Create a large Excel file with sample sales data
np.random.seed(42)
data = {
    'Date': pd.date_range('2023-01-01', periods=1000, freq='h'),
    'Product': np.random.choice(['Widget A', 'Widget B', 'Widget C', 'Gadget X', 'Gadget Y'], 1000),
    'Sales': np.random.randint(1, 1000, 1000),
    'Revenue': np.random.uniform(10.0, 5000.0, 1000),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], 1000),
    'Customer_ID': np.random.randint(1000, 9999, 1000)
}

df = pd.DataFrame(data)
df.to_excel('large_sales_data.xlsx', index=False)

# Create a sample chart image
img = Image.new('RGB', (600, 500), color='white')
draw = ImageDraw.Draw(img)

# Draw a simple bar chart
products = ['Widget A', 'Widget B', 'Widget C', 'Gadget X', 'Gadget Y']
values = [250, 180, 320, 150, 280]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

max_value = max(values)
bar_width = 120
start_x = 100

for i, (product, value, color) in enumerate(zip(products, values, colors)):
    x = start_x + i * (bar_width + 20)
    height = int((value / max_value) * 400)
    y = 500 - height

    # Draw bar
    draw.rectangle([x, y, x + bar_width, 500], fill=color)

    # Add labels (simplified without font)
    draw.text((x + 10, 510), product[:8], fill='black')
    draw.text((x + 10, y - 20), str(value), fill='black')

draw.text((300, 50), 'Sales Performance by Product', fill='black')
img.save('sales_chart.png')

# Check file sizes
excel_size = os.path.getsize('large_sales_data.xlsx') / (1024 * 1024)  # MB
image_size = os.path.getsize('sales_chart.png') / (1024 * 1024)  # MB

print(f"Excel file size: {excel_size:.2f} MB")
print(f"Image file size: {image_size:.2f} MB")
print(f"Total payload size: {excel_size + image_size:.2f} MB")

Excel file size: 0.05 MB
Image file size: 0.01 MB
Total payload size: 0.05 MB


## Create Multi-Modal Agent

Let's create an agent that can process both Excel files and images from large payloads:

In [3]:
%%writefile multimodal_data_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
import pandas as pd
import base64
import io
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# Initialize the model and agent
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
    max_tokens=16000
)

agent = Agent(
    model=model,
    system_prompt="""
    You are a data analysis assistant that can process large Excel files and images.
    When given multi-modal data, analyze both the structured data and visual content,
    then provide comprehensive insights combining both data sources.
    """
)

@app.entrypoint
def multimodal_data_processor(payload, context):
    """
    Process large multi-modal payloads containing Excel data and images.

    Args:
        payload: Contains prompt, excel_data (base64), image_data (base64)
        context: Runtime context information

    Returns:
        str: Analysis results from both data sources
    """
    prompt = payload.get("prompt", "Analyze the provided data.")
    excel_data = payload.get("excel_data", "")
    image_data = payload.get("image_data", "")

    print(f"=== Large Payload Processing ===")
    print(f"Session ID: {context.session_id}")

    if excel_data:
        print(f"Excel data size: {len(excel_data) / 1024 / 1024:.2f} MB")
    if image_data:
        print(f"Image data size: {len(image_data) / 1024 / 1024:.2f} MB")
    print(f"Excel data {excel_data}")
    print(f"Image data {image_data}")
    print(f"=== Processing Started ===")
    # Decode base64 to bytes
    excel_bytes = base64.b64decode(excel_data)
    # Decode base64 to bytes
    image_bytes = base64.b64decode(image_data)

    # Enhanced prompt with data context
    enhanced_prompt = f"""{prompt}
    Please analyze both data sources and provide insights.
    """

    response = agent(
        [{
            "document": {
                "format": "xlsx",
                "name": "excel_data",
                "source": {
                    "bytes": excel_bytes
                }
            }
        },
        {
            "image": {
                "format": "png",
                "source": {
                    "bytes": image_bytes
                }
            }
        },
        {
            "text": enhanced_prompt
        }]
    )
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Writing multimodal_data_agent.py


## Setup Infrastructure and Deploy Agent

In [4]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="multimodal_data_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="multimodal_data_agent_" + user_name
)

launch_result = agentcore_runtime.launch()

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/03-handling-large-payloads/multimodal_data_agent.py, bedrock_agentcore_name=multimodal_data_agent
Configuring BedrockAgentCore agent: multimodal_data_agent_eric_fu
Generated .dockerignore
Generated Dockerfile: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/03-handling-large-payloads/Dockerfile
Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/03-handling-large-payloads/.dockerignore
Setting 'multimodal_data_agent_eric_fu' as default agent
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/03-handling-large-payloads/.bedrock_agentcore.yaml
🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with Co

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-multimodal_data_agent_eric_fu


Getting or creating execution role for agent: multimodal_data_agent_eric_fu
Using AWS region: us-west-2, account ID: 372080370602
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1f76a6bfa5
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1f76a6bfa5
Starting execution role creation process for agent: multimodal_data_agent_eric_fu
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1f76a6bfa5
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-1f76a6bfa5
✓ Role created: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-1f76a6bfa5
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-multimodal_data_agent_eric_fu
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-1f76a6bfa5
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for

In [5]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Deployment status: {status}")

print(f"Final status: {status}")

Retrieved Bedrock AgentCore status for: multimodal_data_agent_eric_fu


Final status: READY


## Test Large Multi-Modal Payloads

Now let's test the agent with large payloads containing both Excel data and images:

In [6]:
import base64
import uuid
import json
from IPython.display import Markdown, display

# Encode files to base64
with open('large_sales_data.xlsx', 'rb') as f:
    excel_base64 = base64.b64encode(f.read()).decode('utf-8')

with open('sales_chart.png', 'rb') as f:
    image_base64 = base64.b64encode(f.read()).decode('utf-8')

# Create large payload
large_payload = {
    "prompt": "Analyze the sales data from the Excel file and correlate it with the chart image. Provide insights on sales performance and trends.",
    "excel_data": excel_base64,
    "image_data": image_base64
}

session_id = str(uuid.uuid4())
print(f"📊 Processing large multi-modal payload...")
print(f"📋 Session ID: {session_id}")
print(f"📄 Excel size: {len(excel_base64) / 1024 / 1024:.2f} MB")
print(f"🖼️ Image size: {len(image_base64) / 1024 / 1024:.2f} MB")
print(f"📦 Total payload: {len(json.dumps(large_payload)) / 1024 / 1024:.2f} MB\n")

# Invoke agent with large payload
invoke_response = agentcore_runtime.invoke(
    large_payload,
    session_id=session_id
)
final_response = ""
for r in invoke_response['response']:
    final_response += r
response_data = final_response
display(Markdown(response_data))

📊 Processing large multi-modal payload...
📋 Session ID: 8d7702bb-69f2-4771-9ed0-035b2fdb9b48
📄 Excel size: 0.06 MB
🖼️ Image size: 0.01 MB
📦 Total payload: 0.07 MB



"# Sales Data Analysis: Excel Data + Chart Correlation\n\n## Data Overview\n- **Time Period:** January 1 - February 11, 2023\n- **Total Records:** 2,880+ transactions\n- **Products:** Widget A, Widget B, Widget C, Gadget X, Gadget Y\n- **Regions:** East, West, North, South\n- **Metrics:** Sales units, Revenue, Customer IDs\n\n---\n\n## Key Findings\n\n### 1. **Sales Performance by Product** (Correlating with Chart)\n\nThe chart shows \"Sales Performance by Product\" with the following approximate values:\n- **Gadget X: ~250 units** (Red bar)\n- **Widget C: ~180 units** (Teal bar)\n- **Widget A: ~320 units** (Blue bar - highest)\n- **Widget B: ~150 units** (Green bar - lowest)\n\n**Analysis:** Widget A emerges as the top performer, followed by Gadget X. Widget B shows the lowest sales volume, suggesting potential optimization opportunities.\n\n### 2. **Revenue Insights**\n\nFrom the data sample:\n- Average transaction revenue: **~$2,400-$2,500**\n- Revenue range: **$10.94 to $4,985.42**\n- Wide variance indicates significant price differentiation or quantity variations\n\n**Top Revenue Transactions:**\n- Widget A entries frequently exceed $4,800\n- Gadget X shows volatility with some high-value transactions (>$4,800)\n\n### 3. **Regional Distribution**\n\nAll four regions are represented throughout the dataset:\n- **East, West, North, South** - Relatively balanced customer base\n- No single region dominates the sales volume\n- This suggests market diversification, which reduces regional risk\n\n### 4. **Product-Specific Observations**\n\n| Product | Characteristics |\n|---------|-----------------|\n| **Widget A** | Highest sales volume (320 units); consistent performer |\n| **Gadget X** | Strong second (250 units); variable revenue per unit |\n| **Widget C** | Moderate performer (180 units); stable revenue |\n| **Widget B** | Lowest volume (150 units); needs promotion/investigation |\n| **Gadget Y** | Not shown in chart; appears frequently in data; mixed performance |\n\n### 5. **Temporal Trends**\n\n- Data spans **41 days** (Jan 1 - Feb 11)\n- **24-hour transaction cycle** per day suggests continuous operations\n- No obvious daily seasonality patterns in the sample\n- Customer IDs range from 1,000-9,999 (suggests ~9,000 unique customers)\n\n### 6. **Performance Gaps & Opportunities**\n\n**Strengths:**\n- Widget A's dominance indicates strong market demand\n- Diversified product portfolio spreads risk\n- Consistent daily sales volume\n\n**Concerns:**\n- Widget B's low performance (150 units) vs. Widget A (320 units) = 53% gap\n- High revenue volatility suggests inconsistent pricing/margins\n- Need to understand why Widget B underperforms\n\n---\n\n## Recommendations\n\n1. **Investigate Widget B Underperformance**\n   - Analyze product quality, pricing, or market positioning\n   - Consider promotional campaigns\n\n2. **Optimize Widget A Supply Chain**\n   - Ensure stock levels meet demand\n   - Explore margin optimization\n\n3. **Reduce Revenue Volatility**\n   - Standardize pricing strategies\n   - Analyze cost structure by product\n\n4. **Regional Analysis**\n   - Deep-dive into region-specific trends\n   - Identify regional growth opportunities\n\n5. **Customer Segmentation**\n   - Analyze repeat purchase patterns\n   - Target high-value customers\n\n---\n\n## Data Quality Notes\n- Transactions appear randomly distributed across regions\n- No missing values in the sample analyzed\n- Timestamps show hourly granularity (excellent for trend analysis)"

## Cleanup Resources

In [7]:
import boto3

# Clean up AWS resources
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Delete AgentCore Runtime
runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)

# Delete ECR repository
ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Clean up local files
os.remove('large_sales_data.xlsx')
os.remove('sales_chart.png')

print("✅ Cleanup completed!")

✅ Cleanup completed!


# Congratulations!

You have successfully demonstrated handling large multi-modal payloads with Amazon Bedrock AgentCore Runtime!

## What you've learned:

### Large Payload Processing
* **100MB Support**: AgentCore Runtime can handle payloads up to 100MB
* **Base64 Encoding**: Secure transmission of binary data through JSON payloads
* **Efficient Processing**: Runtime optimized for large data processing

### Multi-Modal Capabilities
* **Excel Analysis**: Processing structured data from spreadsheets
* **Image Processing**: Analyzing visual content and charts
* **Combined Analysis**: Correlating insights from multiple data types

### Key Benefits
* **Rich Data Processing**: Handle complex, multi-format datasets
* **Scalable Architecture**: Runtime designed for large workloads
* **Tool Integration**: Custom tools for specialized data processing
* **Enterprise Ready**: Secure handling of sensitive business data

This demonstrates AgentCore Runtime's capability to handle enterprise-scale data processing tasks with multiple data modalities, making it ideal for complex business intelligence and data analysis applications.